# 13 — Tensor- and pipeline-parallel fulfilment (TP=1, TP=2, PP=2)

**UNEXECUTED — Kaggle verification pending.** Stokastisk 0.1.0 features are verified only when *you* run this notebook on Kaggle (Accelerator **GPU T4 x2**, Internet ON, Python 3.13). Statuses: PASS / FAIL / ERROR / WARNING / SKIPPED / NOT_RUN. A feature that is unsupported is reported as such; it is never turned into a PASS.

Re-verifies the parallel layouts on the current wheel, one engine per process: TP=1 (one GPU busy), TP=2 and PP=2 (BOTH GPUs hold a shard), parallel_config as reported by vLLM, deterministic non-empty output, TRITON_ATTN only, no Flash libraries, clean shutdown and memory release. Then benchmarks all three on the same workload. Unlike notebook 08, a PP=2 or TP=2 failure FAILS this notebook.

## 1. Locate the release bundle

In [1]:
import glob, hashlib, json, os, pathlib, shutil, sys, time, urllib.request, zipfile
from concurrent.futures import ThreadPoolExecutor
import hashlib, json, os, pathlib, shutil, sys, time, urllib.request, zipfile
from concurrent.futures import ThreadPoolExecutor

HF_REPO = "waqasm86/stokastisk"
HF_BASE = f"https://huggingface.co/{HF_REPO}/resolve"


def _open(url, headers=None):
    return urllib.request.urlopen(urllib.request.Request(url, headers=headers or {"User-Agent": "stokastisk-fetch/1"}), timeout=60)


def _range_ok(url):
    if not url.startswith("http"):
        return None
    with _open(url, {"Range": "bytes=0-0", "User-Agent": "stokastisk-fetch/1"}) as r:
        cr = r.headers.get("Content-Range")                      # "bytes 0-0/37080766"
        return int(cr.rsplit("/", 1)[1]) if r.status == 206 and cr else None


def _download(url, dest, workers=1, retries=3):
    """Download to `<dest>.part`, check free disk space and the advertised length, then replace `dest` atomically; the partial file is removed on any failure."""
    dest = pathlib.Path(dest); part = dest.with_name(dest.name + ".part")
    part.unlink(missing_ok=True)
    try:
        total = _range_ok(url)
        if total and shutil.disk_usage(dest.parent).free < total + (64 << 20):
            raise RuntimeError(f"not enough free disk space in {dest.parent} for {total / 1e6:.0f} MB")
        _download_to(url, part, workers, retries)
        if total and part.stat().st_size != total:
            raise RuntimeError(f"incomplete download: {part.stat().st_size} of {total} bytes")
        os.replace(part, dest)
    except BaseException:
        part.unlink(missing_ok=True)
        raise


def _download_to(url, dest, workers=1, retries=3):
    dest = pathlib.Path(dest); total = _range_ok(url)
    t0 = time.time()
    if total and total > (4 << 20):
        chunk = -(-total // workers); parts = [(i * chunk, min(total, (i + 1) * chunk) - 1) for i in range(workers) if i * chunk < total]
        with open(dest, "wb") as f:
            f.truncate(total)

        def part(rng):
            a, b = rng
            for attempt in range(retries):
                try:
                    with _open(url, {"Range": f"bytes={a}-{b}", "User-Agent": "stokastisk-fetch/1"}) as r, open(dest, "r+b") as f:
                        f.seek(a); shutil.copyfileobj(r, f, 1 << 20)
                    return
                except Exception:
                    if attempt == retries - 1: raise
                    time.sleep(2 ** attempt)
        with ThreadPoolExecutor(len(parts)) as ex:
            list(ex.map(part, parts))
        mode = f"{len(parts)} parallel ranges"
    else:
        for attempt in range(retries):
            try:
                with _open(url) as r, open(dest, "wb") as f:
                    shutil.copyfileobj(r, f, 1 << 20)
                break
            except Exception:
                if attempt == retries - 1: raise
                time.sleep(2 ** attempt)
        mode = "single stream"
    n = dest.stat().st_size
    print(f"downloaded {n / 1e6:.1f} MB in {time.time() - t0:.1f}s ({n / 1e6 / max(time.time() - t0, 1e-6):.1f} MB/s, {mode})")


def _sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for c in iter(lambda: f.read(1 << 20), b""):
            h.update(c)
    return h.hexdigest()


def fetch_release(release, dest="/tmp/stokastisk-release", base=None, revision="main", workers=1):
    """Return the verified, extracted bundle directory (…/<release>). `base` overrides the Hub URL prefix (tests use file://)."""
    base = (base or HF_BASE).rstrip("/")
    root = f"{base}/{revision}/releases/{release}" if base.startswith("http") else f"{base}/releases/{release}"
    dest = pathlib.Path(dest); dest.mkdir(parents=True, exist_ok=True)
    zname = f"{release}-kaggle-input.zip"; zpath = dest / zname
    if zpath.exists(): zpath.unlink()
    with _open(f"{root}/{zname}.sha256") as r:
        expected = r.read().decode().split()[0]
    _download(f"{root}/{zname}", zpath, workers=workers)
    got = _sha256(zpath)
    if got != expected:
        zpath.unlink(); raise RuntimeError(f"SHA256 mismatch for {zname}: expected {expected}, got {got}")
    out = dest / release
    if out.exists(): shutil.rmtree(out)
    with zipfile.ZipFile(zpath) as z:
        assert z.testzip() is None
        for n in z.namelist():
            if n.startswith("/") or ".." in pathlib.PurePosixPath(n).parts or not n.startswith(release + "/"):
                raise RuntimeError(f"unsafe or foreign path in archive: {n}")
        z.extractall(dest)
    m = json.loads((out / "manifests" / "RELEASE-MANIFEST.json").read_text())
    if m.get("name") != release:
        raise RuntimeError(f"manifest name {m.get('name')!r} != {release!r}")
    print(f"release {release}: SHA256 {got[:16]}... verified, extracted to {out}")
    return out

# Only the bundle directory named RELEASE whose RELEASE-MANIFEST.json carries the same name is accepted (never an older rc).
RELEASE = "stokastisk-0.1.0-kaggle-t4-sm75"
cands = [pathlib.Path(p).parent.parent for p in glob.glob("/kaggle/input/**/manifests/SHA256SUMS", recursive=True)]
if os.environ.get("STOKASTISK_BUNDLE"): cands = [pathlib.Path(os.environ["STOKASTISK_BUNDLE"])]
match = [c for c in cands if c.name == RELEASE]
if not match:
    print(f"{RELEASE} is not attached (attached: {sorted(str(c) for c in cands)}); downloading from Hugging Face (needs Internet ON and a published release) ...")
    match = [fetch_release(RELEASE)]
assert len({(c / "manifests" / "SHA256SUMS").read_bytes() for c in match}) == 1, f"several DIFFERENT copies of {RELEASE}: {match}"
BUNDLE = match[0]
_m = json.loads((BUNDLE / "manifests" / "RELEASE-MANIFEST.json").read_text())
assert _m.get("name") == RELEASE, f"manifest name {_m.get('name')!r} != expected {RELEASE!r}"
SCRIPTS = BUNDLE / "scripts" / "kaggle"
os.environ["STOKASTISK_OUT"] = "/kaggle/working/stokastisk-acceptance"
print("release:", RELEASE, "| bundle:", BUNDLE, "| manifest sha256:", hashlib.sha256((BUNDLE / "manifests" / "SHA256SUMS").read_bytes()).hexdigest())

stokastisk-0.1.0-kaggle-t4-sm75 is not attached (attached: []); downloading from Hugging Face (needs Internet ON and a published release) ...
downloaded 37.2 MB in 4.7s (7.8 MB/s, 1 parallel ranges)
release stokastisk-0.1.0-kaggle-t4-sm75: SHA256 4643961225d0cd61... verified, extracted to /tmp/stokastisk-release/stokastisk-0.1.0-kaggle-t4-sm75
release: stokastisk-0.1.0-kaggle-t4-sm75 | bundle: /tmp/stokastisk-release/stokastisk-0.1.0-kaggle-t4-sm75 | manifest sha256: 97ad66602c44100398a52c8a024dd66abedc75501ecbff6e60f26e8e499709b7


## 2. Install into this fresh session (`--no-deps`, protected packages untouched)

In [2]:
import subprocess, sys, os
r = subprocess.run([sys.executable, str(SCRIPTS / "02_install.py"), str(BUNDLE)],
                   env=dict(os.environ, STOKASTISK_OUT="/kaggle/working/stokastisk-acceptance/prereq-install"))
assert r.returncode == 0, "prerequisite install failed (see output above); do not continue"

PASS SHA256SUMS verified before any install -- all files match
PASS python 3.13 -- 3.13.16
PASS baseline protected (torch/triton exact, transformers>=5.5.3, no prohibited pkgs) -- {'torch': '2.11.0+cu130', 'triton': '3.6.0', 'transformers': '5.20.2', 'numpy': '1.26.4', 'nvidia-cuda-nvcc': '13.0.88', 'nvidia-nvvm': '13.0.88'}
PASS two Tesla T4 (SM75) -- 2 gpu(s) [(7, 5), (7, 5)]
PASS exactly two wheels -- ['stokastisk-0.1.0-py3-none-any.whl', 'vllm-0.25.1-cp38-abi3-linux_x86_64.whl']
PASS wheels contain no FlashAttention/FlashInfer/FlashMLA files -- none
PASS dependency plan has no conflicts (nothing is installed if this fails) -- {'install': {'apache-tvm-ffi': '0.1.9', 'xgrammar': '0.2.5.post1', 'llguidance': '1.7.6', 'cbor2': '6.2.0.post2', 'msgspec': '0.22.0', 'openai-harmony': '0.0.8', 'partial-json-parser': '0.2.1.1.post7', 'pybase64': '1.5.1'}}
PASS download exact pins (no resolver, --no-deps) -- 8 pins
PASS install wheels --force-reinstall --no-deps (replaces any other stokastisk

## 3. Provenance: scripts and wheels must match `manifests/SHA256SUMS`

In [3]:
import hashlib
sha = lambda p: hashlib.sha256(open(p, "rb").read()).hexdigest()
for w in sorted((BUNDLE / "wheels").glob("*.whl")): print(f"{sha(w)}  {w.name}")
expected = {l.split(None, 1)[1].strip(): l.split()[0] for l in (BUNDLE / "manifests" / "SHA256SUMS").read_text().splitlines()}
for rel in [f"scripts/kaggle/{s.name}" for s in SCRIPTS.glob("*.py")] + [f"wheels/{w.name}" for w in (BUNDLE / "wheels").glob("*.whl")]:
    assert expected.get(rel) == sha(BUNDLE / rel), f"{rel} does not match manifests/SHA256SUMS"
print("all executed scripts and wheels match manifests/SHA256SUMS")

29edaa550f49a1cd343a8f043ddd2b5c98c9154ab1da0172fc3fc8725aa3ee49  stokastisk-0.1.0-py3-none-any.whl
cc3bcafb6516791990ca302073ae153a08465d3df1e460d666e3ec09b4e98f5e  vllm-0.25.1-cp38-abi3-linux_x86_64.whl
all executed scripts and wheels match manifests/SHA256SUMS


## 4. Installed packages must be byte-identical to the bundle

In [4]:
import importlib, importlib.metadata as md, zipfile
for w in sorted((BUNDLE / "wheels").glob("*.whl")):
    pkg = w.name.split("-")[0]; mod = importlib.import_module(pkg); root = pathlib.Path(mod.__file__).resolve().parent
    z = zipfile.ZipFile(w); bad = []
    for n in z.namelist():
        if n.startswith(pkg + "/") and n.endswith((".py", ".so")):
            f = root.parent / n
            if not f.is_file() or hashlib.sha256(f.read_bytes()).hexdigest() != hashlib.sha256(z.read(n)).hexdigest(): bad.append(n)
    print(f"{pkg} {md.version(pkg)}: {'identical to ' + w.name if not bad else 'MISMATCH ' + str(bad)}")
    assert not bad, f"installed {pkg} differs from the bundle wheel {w.name}"

stokastisk 0.1.0: identical to stokastisk-0.1.0-py3-none-any.whl
vllm 0.25.1: identical to vllm-0.25.1-cp38-abi3-linux_x86_64.whl


## 5. Stages

In [5]:
import subprocess, sys, os
def run_stage(script, *args, timeout=1800, log=None):
    """Run a bundled stage script with a finite timeout; returns its exit code (124 = timeout). Failures are judged in the verdict cell."""
    env = dict(os.environ, PYTHONPATH=str(SCRIPTS) + ":" + os.environ.get("PYTHONPATH", ""), VLLM_LOGGING_LEVEL="INFO")
    out = open("/kaggle/working/stokastisk-acceptance/" + (log or script + ".log"), "w") if log is not False else None
    try:
        r = subprocess.run([sys.executable, str(SCRIPTS / script), *args], cwd=SCRIPTS, env=env, timeout=timeout, stdout=out, stderr=subprocess.STDOUT if out else None)
        rc = r.returncode
    except subprocess.TimeoutExpired:
        rc = 124
    print(f"{script} {' '.join(args)} exit code: {rc}")
    return rc
os.makedirs("/kaggle/working/stokastisk-acceptance", exist_ok=True)

In [6]:
RC = {}
M = "Qwen/Qwen2.5-1.5B-Instruct"
for mode in ("tp1", "tp2", "pp2"):
    RC["15_fulfil_" + mode] = run_stage("dual_15_tp_pp_fulfilment.py", mode, M, timeout=1500, log=f"15_fulfil_{mode}.log")
RC["15_fulfil_summary"] = run_stage("dual_15_tp_pp_fulfilment.py", "summary", timeout=120, log="15_fulfil_summary.log")

dual_15_tp_pp_fulfilment.py tp1 Qwen/Qwen2.5-1.5B-Instruct exit code: 0
dual_15_tp_pp_fulfilment.py tp2 Qwen/Qwen2.5-1.5B-Instruct exit code: 0
dual_15_tp_pp_fulfilment.py pp2 Qwen/Qwen2.5-1.5B-Instruct exit code: 0
dual_15_tp_pp_fulfilment.py summary exit code: 0


### Same workload on all three layouts (every repeat recorded)

In [7]:
for mode in ("tp1", "tp2", "pp2"):
    RC["bench_" + mode] = run_stage("dual_bench.py", mode, M, "standard", timeout=1800, log=f"bench_{mode}.log")

dual_bench.py tp1 Qwen/Qwen2.5-1.5B-Instruct standard exit code: 0
dual_bench.py tp2 Qwen/Qwen2.5-1.5B-Instruct standard exit code: 0
dual_bench.py pp2 Qwen/Qwen2.5-1.5B-Instruct standard exit code: 0


## Package evidence → `/kaggle/working/13-stokastisk-acceptance.zip`

In [8]:
import shutil
print(shutil.make_archive("/kaggle/working/13-stokastisk-acceptance", "zip", "/kaggle/working", "stokastisk-acceptance"))

/kaggle/working/13-stokastisk-acceptance.zip


## Verdict (fails the notebook if a REQUIRED stage failed)

All seven stages are required. A slower TP=2 / PP=2 is a measurement, not a failure; a crash, an idle second GPU or an empty output is.

In [9]:
required = ['15_fulfil_tp1', '15_fulfil_tp2', '15_fulfil_pp2', '15_fulfil_summary', 'bench_tp1', 'bench_tp2', 'bench_pp2']
bad = {k: v for k, v in RC.items() if k in required and v != 0}
print('all stage exit codes:', RC)
assert not bad, f'FAILED required stage(s): {bad}'

all stage exit codes: {'15_fulfil_tp1': 0, '15_fulfil_tp2': 0, '15_fulfil_pp2': 0, '15_fulfil_summary': 0, 'bench_tp1': 0, 'bench_tp2': 0, 'bench_pp2': 0}
